# Simple Trend-Following Alpha (SMA Crossover)

This notebook demonstrates a very basic, short, and easy-to-understand backtesting strategy: a Simple Moving Average (SMA) crossover. It's meant for clients seeking a quick example of how to implement and evaluate an alpha signal using Polars on the Binance Perps Panel.

In [ ]:
import sys
import polars as pl
import matplotlib.pyplot as plt

sys.path.append('..')
from loader import load_panel

# Ensure plots show up inline
%matplotlib inline
plt.style.use('ggplot')

## 1. Data Loading and Feature Engineering

We load BTCUSDT data and compute two moving averages: a fast SMA (e.g., 24 hours) and a slow SMA (e.g., 168 hours or 1 week).

In [ ]:
# Load a subset of the panel for BTCUSDT in 2023
lf = load_panel(start_date="2023-01-01", end_date="2024-01-01", symbols=["BTCUSDT"])

# Define parameters for the Simple Moving Averages
FAST_MA_PERIOD = 24  # 24 hours (1 day)
SLOW_MA_PERIOD = 168 # 168 hours (7 days)

# Compute SMAs and our position signal
# Signal = +1 (Long) when Fast > Slow
# Signal = -1 (Short) when Fast < Slow
lf_alpha = (
    lf
    .sort(["symbol", "timestamp"])
    .with_columns([
        pl.col("close").rolling_mean(window_size=FAST_MA_PERIOD).alias("sma_fast"),
        pl.col("close").rolling_mean(window_size=SLOW_MA_PERIOD).alias("sma_slow")
    ])
    .with_columns(
        pl.when(pl.col("sma_fast") > pl.col("sma_slow")).then(pl.lit(1.0))
        .when(pl.col("sma_fast") < pl.col("sma_slow")).then(pl.lit(-1.0))
        .otherwise(pl.lit(0.0))
        .alias("target_position")
    )
    .drop_nulls(subset=["sma_slow"])  # Drop initial burn-in period
)

df_alpha = lf_alpha.collect()
print(df_alpha.select(["timestamp", "close", "sma_fast", "sma_slow", "target_position"]).head())

## 2. Evaluate Strategy Returns

To prevent lookahead bias, we multiply the `target_position` of the *current* hour by the `fwd_ret_1h` (which is the return realized over the *next* hour). We do not include transaction costs or funding rates here to keep the example as simple as possible.

In [ ]:
# Calculate strategy returns (Position * Forward Return)
df_backtest = (
    df_alpha
    .with_columns([
        (pl.col("target_position") * pl.col("fwd_ret_1h")).alias("strat_return")
    ])
    # Calculate cumulative returns
    .with_columns([
        (pl.col("fwd_ret_1h") + 1).cum_prod().alias("cum_bh_return"),    # Buy & Hold
        (pl.col("strat_return") + 1).cum_prod().alias("cum_strat_return") # Strategy
    ])
)

# Extract for plotting
pdf = df_backtest.to_pandas()
pdf.set_index("timestamp", inplace=True)

# Final stats
strat_tot = pdf["cum_strat_return"].iloc[-2] - 1  # Using -2 because fwd_ret is null on very last row
bh_tot = pdf["cum_bh_return"].iloc[-2] - 1

print(f"Strategy Total Return: {strat_tot * 100:.2f}%")
print(f"Buy & Hold Total Return: {bh_tot * 100:.2f}%")

## 3. Plotting the Equity Curve

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(pdf.index, pdf["cum_bh_return"], label="Buy & Hold BTC", color="gray", alpha=0.7)
plt.plot(pdf.index, pdf["cum_strat_return"], label="SMA Crossover Strategy", color="blue")
plt.title("Simple SMA Crossover Backtest (BTCUSDT)")
plt.ylabel("Cumulative Growth (1 = 100%)")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()